In [1]:
!pip -q install -U groq gradio

In [2]:
import os, json, re
from getpass import getpass
from datetime import date
import gradio as gr
from groq import Groq

# Securely request the key at runtime; it is not saved in this notebook.
GROQ_API_KEY = getpass("Enter your NEW Groq API key: ").strip()
if not GROQ_API_KEY:
    raise ValueError("A Groq API key is required.")
client = Groq(api_key=GROQ_API_KEY)

# Change this if Groq retires or changes model availability.
MODEL = "llama-3.3-70b-versatile"

SYSTEM_PROMPT = """You are HealthMate, a cautious nutrition and general-wellness assistant.
Give general educational information, not diagnosis or treatment. Never prescribe medicines,
supplements, or dosages. Do not claim food estimates are exact. Ask for portion size when needed.
For medical symptoms, recommend appropriate professional assessment. For chest pain, severe
breathing difficulty, stroke signs, severe allergic reaction, loss of consciousness, or other
potential emergencies, tell the user to contact local emergency services immediately.
Avoid extreme dieting, crash diets, and unsafe weight-loss advice. Be respectful and concise.
Mention uncertainty and individual variation. Do not invent lab results or clinical facts."""

def ask_ai(message, history=None, extra_context=""):
    if not message or not message.strip():
        return "Please enter a question."
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    if extra_context:
        messages.append({"role": "system", "content": "User-provided context (not verified): " + extra_context})
    # Keep a short conversation history for continuity.
    for item in (history or [])[-8:]:
        if isinstance(item, dict) and item.get("role") in ("user", "assistant"):
            messages.append({"role": item["role"], "content": str(item.get("content", ""))})
    messages.append({"role": "user", "content": message})
    try:
        result = client.chat.completions.create(
            model=MODEL, messages=messages, temperature=0.3, max_tokens=900
        )
        return result.choices[0].message.content or "I couldn't generate a response."
    except Exception as e:
        return f"API error: {type(e).__name__}: {e}"

def calculate_health(age, sex, height_cm, weight_kg, activity, goal):
    try:
        age = int(age); h = float(height_cm); w = float(weight_kg)
        if not (15 <= age <= 100 and 100 <= h <= 230 and 25 <= w <= 350):
            return "Please enter plausible values (age 15–100, height 100–230 cm, weight 25–350 kg)."
        bmi = w / ((h/100) ** 2)
        # Mifflin-St Jeor estimate; sex input is used only for this equation.
        if sex == "Male":
            bmr = 10*w + 6.25*h - 5*age + 5
        elif sex == "Female":
            bmr = 10*w + 6.25*h - 5*age - 161
        else:
            return "Choose Male or Female for the standard Mifflin-St Jeor estimate. This equation may not fit everyone."
        factors = {"Sedentary (little exercise)":1.2, "Light (1–3 days/week)":1.375,
                   "Moderate (3–5 days/week)":1.55, "Very active (6–7 days/week)":1.725,
                   "Extra active (physical job/intense training)":1.9}
        tdee = bmr * factors[activity]
        if goal == "Lose weight gradually":
            target = tdee - 300
            goal_note = "A modest estimated deficit; avoid aggressive restriction."
        elif goal == "Gain weight gradually":
            target = tdee + 250
            goal_note = "A modest estimated surplus; combine with appropriate strength training."
        else:
            target = tdee
            goal_note = "Estimated maintenance intake."
        # Broad adult BMI categories; BMI is a screening measure, not a diagnosis.
        if bmi < 18.5: category = "Below the usual adult BMI range"
        elif bmi < 25: category = "Usual adult BMI range"
        elif bmi < 30: category = "Above the usual adult BMI range"
        else: category = "High adult BMI range"
        return (f"### Your estimates\n"
                f"- **BMI:** {bmi:.1f} — {category}\n"
                f"- **BMR:** {bmr:.0f} kcal/day (estimated resting energy use)\n"
                f"- **TDEE:** {tdee:.0f} kcal/day (estimated maintenance)\n"
                f"- **Daily calorie target:** {target:.0f} kcal/day\n\n"
                f"{goal_note}\n\n"
                "**Notes:** These are population-based estimates, not measurements. BMI does not distinguish muscle from fat and is not suitable for every person. Energy needs vary; seek individualized advice if pregnant, under 18, managing a medical condition, or with a history of disordered eating.")
    except Exception as e:
        return f"Please check your entries. ({e})"

def estimate_food(food_text):
    if not food_text or not food_text.strip():
        return "Describe the food and portion, e.g. '2 boiled eggs and 1 medium banana'."
    prompt = f"""Estimate nutrition for this food description: {food_text}
Return a compact table with columns Food/portion, Calories (kcal), Protein (g), Carbs (g), Fat (g).
Then give TOTAL calories, protein, carbs and fat. State assumptions about portion sizes and clearly
label values as approximate. If quantity is missing, state the assumed portion. Do not invent precision.
Do not provide medical advice."""
    return ask_ai(prompt)

def add_food(food_text, log):
    log = list(log or [])
    if not food_text or not food_text.strip():
        return log, log_summary(log), "Enter a food and portion first."
    estimate = estimate_food(food_text)
    # Store the description and estimate; numeric totals are not guessed or parsed from free text.
    log.append({"food": food_text.strip(), "estimate": estimate})
    return log, log_summary(log), estimate

def log_summary(log):
    if not log:
        return "No meals logged yet."
    return "\n\n".join([f"**{i+1}. {x['food']}**\n{x['estimate']}" for i,x in enumerate(log)])

def clear_log():
    return [], "No meals logged yet.", "Daily log cleared."

def nutrition_chat(message, history):
    return ask_ai(message, history)


Enter your NEW Groq API key: ··········


In [3]:
with gr.Blocks(title="HealthMate AI", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🩺 HealthMate AI\nYour nutrition, calorie and wellness companion")
    gr.Markdown("**Educational estimates only — not a diagnosis or medical service.**")

    with gr.Tab("📊 Health measurements"):
        gr.Markdown("Enter your details to estimate BMI, BMR, TDEE and a calorie target.")
        with gr.Row():
            age = gr.Number(label="Age (years)", value=22, precision=0)
            sex = gr.Dropdown(["Male", "Female"], label="Sex used for BMR equation", value="Male")
        with gr.Row():
            height = gr.Number(label="Height (cm)", value=174)
            weight = gr.Number(label="Weight (kg)", value=80)
        activity = gr.Dropdown(
            ["Sedentary (little exercise)", "Light (1–3 days/week)", "Moderate (3–5 days/week)",
             "Very active (6–7 days/week)", "Extra active (physical job/intense training)"],
            label="Activity level", value="Light (1–3 days/week)")
        goal = gr.Radio(["Lose weight gradually", "Maintain weight", "Gain weight gradually"],
                        label="Goal", value="Lose weight gradually")
        calc_btn = gr.Button("Calculate my estimates", variant="primary")
        health_out = gr.Markdown()
        calc_btn.click(calculate_health, [age, sex, height, weight, activity, goal], health_out)

    with gr.Tab("🍽️ Food calorie estimator"):
        food = gr.Textbox(label="What did you eat?", placeholder="Example: 2 idlis, 1 cup sambar and 1 boiled egg", lines=2)
        estimate_btn = gr.Button("Estimate nutrition", variant="primary")
        food_out = gr.Markdown()
        estimate_btn.click(estimate_food, food, food_out)

    with gr.Tab("🗓️ Daily food log"):
        gr.Markdown("Add a meal to keep a session-only log. AI estimates are approximate and are not automatically converted into verified numeric totals.")
        meal = gr.Textbox(label="Meal and portion", placeholder="Example: 150 g cooked rice, 100 g chicken curry")
        add_btn = gr.Button("Add meal", variant="primary")
        clear_btn = gr.Button("Clear today's log")
        log_state = gr.State([])
        log_out = gr.Markdown("No meals logged yet.")
        latest_out = gr.Markdown()
        add_btn.click(add_food, [meal, log_state], [log_state, log_out, latest_out])
        clear_btn.click(clear_log, None, [log_state, log_out, latest_out])

    with gr.Tab("💬 Healthcare & wellness AI"):
        chatbot = gr.ChatInterface(
            fn=nutrition_chat,
            title="Ask HealthMate",
            description="Ask general questions about nutrition, exercise, sleep, hydration and healthy habits. No diagnosis or medication advice.",
            examples=["How can I build a balanced breakfast?", "Explain protein, carbs and fats simply.",
                      "What are some beginner-friendly exercises?", "How can I improve my sleep routine?"],
        )

    gr.Markdown("### Privacy & safety\nAvoid entering identifying information. Do not use this app for emergencies, diagnosis, medication decisions, or treatment. If you have concerning symptoms, contact a qualified healthcare professional.")

demo.launch(share=True, debug=False)


/tmp/ipykernel_3982/2019557414.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="HealthMate AI", theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://83e72a4b94cbc00bec.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
